# 50 Startups Success Prediction

**Tabular Regression Project** — Predict startup profitability from spending data.

Models: CatBoost · LightGBM · XGBoost · TabPFN-v2  
Baselines: LazyPredict · FLAML AutoML  
Dataset: 50 Startups (50 rows × 5 columns)  
Target: `Profit`

## 2 · Project Overview

We predict the **Profit** of 50 US startups based on their spending in three areas — R&D, Administration, and Marketing — plus the state where the company operates.

Because the dataset has only 50 rows, it is an excellent test bed for models designed for small data (TabPFN-v2) versus models that typically need thousands of rows (XGBoost, CatBoost).

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a small tabular dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline linear-regression model and compare against modern boosting/AutoML.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Evaluate regression models with RMSE, MAE, R², and residual analysis.
7. Understand when small-data models (TabPFN-v2) outperform large-dataset models.

## 4 · Problem Statement

Given a startup's spending on **R&D**, **Administration**, and **Marketing**, plus the **State** it operates in, predict its **Profit**.

## 5 · Why This Project Matters

- **Startup valuation** is a common real-world regression problem.
- With only 50 rows this exposes model behavior at the small-data limit.
- It teaches that **more complex models are not always better** on tiny datasets.
- Understanding which spending category drives profit is directly actionable.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 50 |
| **Columns** | 5 |
| **Features** | R&D Spend, Administration, Marketing Spend, State |
| **Target** | Profit (continuous, USD) |
| **Categorical** | State (3 levels: New York, California, Florida) |
| **Missing values** | None |
| **File** | `50_Startups.csv` (included locally) |

## 7 · Dataset Source and License Notes

- **Source**: Classic toy dataset from Kaggle / ML courses.
- **License**: Public / educational use. No PII.
- **Limitations**: Synthetic or heavily simplified — real startup financials have many more features.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")
%matplotlib inline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "Profit"

# Resolve data path
_candidates = ["50_Startups.csv", "Regression/50 Startups Success Prediction/50_Startups.csv"]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), _candidates[0])
SAVE_DIR = os.path.dirname(os.path.abspath(DATA_PATH)) if os.path.exists(DATA_PATH) else "."

print(f"Data path: {DATA_PATH}")
print(f"Save dir : {SAVE_DIR}")

## 11 · Dataset Download or Loading

The dataset `50_Startups.csv` is included in this project folder.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Shape: {df.shape}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
df.describe()

In [ ]:
# Correlation heatmap
num_cols = df.select_dtypes(include="number").columns.tolist()
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot
sns.pairplot(df, diag_kind="kde", plot_kws={"alpha": 0.6, "s": 30})
plt.suptitle("Pairplot of All Features", y=1.02)
plt.show()

In [ ]:
# State distribution
fig, ax = plt.subplots(figsize=(6, 4))
df["State"].value_counts().plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Startups by State")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of Profit.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df[TARGET], bins=15, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel(TARGET)
axes[0].set_ylabel("Frequency")

axes[1].boxplot(df[TARGET], vert=True)
axes[1].set_title(f"Box Plot of {TARGET}")
axes[1].set_ylabel(TARGET)

plt.tight_layout()
plt.show()

print(f"Range: ${df[TARGET].min():,.0f} to ${df[TARGET].max():,.0f}")
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")
print(f"Std: ${df[TARGET].std():,.0f}, Skew: {df[TARGET].skew():.3f}")

## 15 · Train/Validation/Test Split Strategy

With only 50 rows we use **80/20 train/test** (40 train, 10 test). Cross-validation provides more robust estimates.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical
cat_cols = X.select_dtypes(include="object").columns.tolist()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None — no imputation needed.
- **Categorical**: State (3 levels) via OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: None detected.

## 17 · Feature Engineering

We add **Total Spend** to capture overall investment level.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["Total_Spend"] = X_train["R&D Spend"] + X_train["Administration"] + X_train["Marketing Spend"]
X_test["Total_Spend"] = X_test["R&D Spend"] + X_test["Administration"] + X_test["Marketing Spend"]

# Sanitize column names for LightGBM/XGBoost
clean = [c.replace("&", "_and_").replace(" ", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean

print(f"Features ({len(clean)}): {clean}")
X_train.head()

## 18 · Baseline Model

Linear Regression baseline. Profit has a strong linear relationship with R&D Spend.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

print("\nCoefficients:")
for feat, coef in zip(X_train.columns, baseline.coef_):
    print(f"  {feat:20s}: {coef:>12,.4f}")
print(f"  {'Intercept':20s}: {baseline.intercept_:>12,.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60, verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML failed (likely XGBoost 3.x compat issue): {e}")
    print("Using Linear Regression as FLAML fallback.")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost, TabPFN-v2

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    cb = CatBoostRegressor(iterations=200, learning_rate=0.05, depth=6,
                           task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    lg = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
                           device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
    lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
           callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
                             device="cuda", tree_method="hist", verbosity=0,
                             n_jobs=-1, random_state=SEED)
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                  early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost: {e}")
gpu_cleanup()

# TabPFN-v2 (optimized for small datasets)
try:
    from tabpfn import TabPFNRegressor
    t0 = time.perf_counter()
    tpfn = TabPFNRegressor(device="cpu", n_estimators=32)
    tpfn.fit(X_train.values, y_train.values)
    timings["TabPFN-v2"] = time.perf_counter() - t0
    results["TabPFN-v2"] = tpfn.predict(X_test.values)
    print(f"TabPFN-v2 RMSE: {root_mean_squared_error(y_test, results['TabPFN-v2']):,.2f}  ({timings['TabPFN-v2']:.1f}s)")
except Exception as e:
    print(f"TabPFN-v2: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.7, s=50, edgecolors="black")
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.0f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual Profit")
    axes[i].set_ylabel("Predicted Profit")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=8, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.7, s=50, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_res = np.sort(residuals)
axes[2].plot(sorted_res, marker="o", linestyle="-", color="steelblue")
axes[2].axhline(0, color="red", linestyle="--")
axes[2].set_title("Sorted Residuals")
axes[2].set_ylabel("Residual")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **R&D Spend** is the strongest predictor (r ~ 0.97 with Profit).
- **Marketing Spend** has a moderate positive effect.
- **Administration** shows almost no correlation with Profit.
- **State** has negligible predictive power.

**Business takeaway:** Startups investing heavily in R&D tend to be most profitable. R&D Spend alone explains ~95% of profit variance.

**Model takeaway:** On 50 rows, simpler models (Linear Regression) often match or beat complex ensembles.

## 26 · Limitations

1. **Tiny dataset** — 50 rows gives high-variance estimates.
2. **Only 4 features** — real success depends on team, market timing, competition.
3. **Synthetic data** — the near-perfect R&D-Profit correlation suggests artificial data.
4. **No time dimension** — macro-economic conditions are ignored.
5. **Possible leakage** — spending may be contemporaneous with profit.

## 27 · How to Improve This Project

1. Collect more real startup data (Crunchbase, SEC filings).
2. Add features: team size, founding year, industry sector, funding rounds.
3. Frame as time-series: track financials over multiple years.
4. Bootstrap confidence intervals on test metrics.
5. Compare Ridge, Lasso, ElasticNet for feature importance stability.

## 28 · Production Considerations

- Do **not** deploy with only 50 data points for real investment decisions.
- Production needs: data validation, retraining schedules, drift monitoring.
- Output **prediction intervals**, not just point estimates.
- Encapsulate preprocessing in a reproducible pipeline.

## 29 · Common Mistakes

1. Overfitting complex models on 50 rows.
2. Not checking for data leakage.
3. Ignoring the baseline — if LR gets R2~0.95, small improvements are noise.
4. One-hot encoding 3 levels on 50 rows — adds noise.
5. Reporting metrics without confidence intervals on 10 test points.

## 30 · Mini Challenge / Exercises

1. Drop the State column and retrain. Does R² change?
2. Try Ridge and Lasso — which features get penalized?
3. Bootstrap the test set 1000 times and plot R² distribution.
4. Remove Total_Spend — is it redundant with individual spend columns?
5. Use only R&D Spend — how close to the multi-feature model?

## 31 · Final Summary / Key Takeaways

1. **R&D Spend dominates** — explains most Profit variance.
2. **Simple models win on small data** — Linear Regression is competitive.
3. **LazyPredict + FLAML** are great for rapid screening.
4. **TabPFN-v2** is designed for small tabular datasets (n < 10K).
5. **Always start with a baseline** — the gap reveals how much non-linearity exists.
6. **More data > more models** — biggest improvement comes from more records.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))